## SARSA example of Gymnasium Frozen Lake

In [ ]:
USE {
    repositories {
        mavenCentral()
        maven("https://central.sonatype.com/repository/maven-snapshots/")
    }
    dependencies {
        implementation("io.github.kotlinrl:integration:0.1.0-SNAPSHOT")
    }
}
%use dataframe
%use kandy

In [ ]:
import io.github.kotlinrl.core.*
import io.github.kotlinrl.integration.gymnasium.*
import io.github.kotlinrl.integration.gymnasium.GymnasiumEnvs.*
import java.io.*

Let's define our hyper-parameters to control training and learning

In [ ]:
val maxStepsPerEpisode = 1_000
val trainingEpisodes = 300_000
val testEpisodes = trainingEpisodes / 10
val initialEpsilon = 1.0
val epsilonDecayRate = 0.9995
val minEpsilon = 0.05
val initialAlpha = 0.5
val alphaDecayRate = 0.99995
val minAlpha = 0.05
val gamma = 0.99
val nStep = 4
val fileName = "FrozenLakeSARSA.csv"

Creating the following
- Env (FrozenLakeEnv = ```Env<Int, Int, Discrete, Discrete>``` based on the Gymnasium
structure)
    - We use a ```TransformState``` wrapper to change the state from ```Int``` space - to a ```MultiDiscrete``` space - which has observations in ```IntArray```.  Essentially we turn the sequence of cells into rows and columns.
    - The ```MultiDiscrete``` space works perfectly for tabular data like the ```QTable```
- ```EpisodeCallback``` to log starting episodes
- ```StateActionListProvider``` to define the list of Actions for any State.  Blackjack only allows
    - Actions
        - 0: Move left
        - 1: Move down
        - 2: Move right
        - 3: Move up
    - State is now ```IntArray``` based on the ```MultiDiscreate``` state space
- The ```QTable``` used to capture training information
    - QLearning updates the ```QTable``` after each ```Trajectory```

In [ ]:
val env = TransformState(
    env = gymnasium.make<FrozenLakeEnv>(
        FrozenLake_v1, render = true, options = mapOf(
            "is_slippery" to true,
            "map_name" to "8x8"
        )
    ),
    transform = { intArrayOf(it / 8, it % 8) },
    observationSpace = MultiDiscrete(8, 4)
)
val actions = listOf(0, 1, 2, 3)
val trainingQtable = QTable(8, 8, 4, stochastic = true)


Next we create the training Agent using the ExpectedSARSA algorithm
- We use an Epsilon Greedy Policy with a decaying epsilon to encourage convergence (experimenting with a constant epsilon would lead to different results) for the exploration factor
- The Epsilon Greedy Policy randomly chooses a number.
    - If it is less than the epsilon value it uses a Random Policy selection
    - Otherwise it uses a Greedy Policy to select the best action from the ```QTable```

The Trainer uses the env and agent with a max steps per episode and trains for the number of training episodes
- The ```expectedSARSAAgent``` function registers the ```ExpectedSARSA``` algorithm as a ```TrajectoryObserver``` with an ```alpha``` and a ```gamma``` value so that when a ```Trajectory``` completes, the algorithm updates the ```QTable```.  The ```ExpectedSARSA``` also needs the policy probabilities, so we use ```WpsilonSoftPolicy``` as the ```PolicyProbabilities``` implementation
- We also register the episode logger
- We then train for the number of training episodes
- When training completes we save the ```QTable``` for later use

In [ ]:
val trainer = episodicTrainer(
    env = env,
    agent = sarsaAgent(
        id = "training",
        policy = epsilonSoftPolicy(
            stateActionListProvider = { actions },
            epsilon = decayingParameterSchedule(
                initialValue = initialEpsilon,
                minValue = minEpsilon,
                decayRate = epsilonDecayRate
            ),
            qTable = trainingQtable
        ),
        alpha = decayingParameterSchedule(
            initialValue = initialAlpha,
            decayRate = alphaDecayRate,
            minValue = minAlpha),
        gamma = gamma
    ),
    maxStepsPerEpisode = maxStepsPerEpisode,
    callbacks = listOf(
        printEpisodeStart(trainingEpisodes / 10),
        printEpisodeTotalTransitions(trainingEpisodes / 10),
        //printEpisodeOnGoalReached(1.0)
    )
)
println("Starting training")
val training = trainer.train(trainingEpisodes)
trainingQtable.save(fileName)


Once training is complete, we create the following
- A new ```QTable``` with the same shape, and load the training data
- A new test ```Agent``` using a ```GreedyPolicy``` against the ```QTable``` with loaded weights
- The Greedy Policy always chooses the best action from the ```QTable```
    - It performs the best action given the state (essentially the agent's row and column)

We then test for the number of testing episodes to compare the episode results (i.e. the average reward achieved)

In [ ]:
val testingQtable = QTable(4, 4, 4)
testingQtable.load(fileName)

In [ ]:
val recordEnv = RecordVideo(env = env, folder = "videos/frozen_lake_sarsa", testEpisodes / 3)
val tester = episodicTrainer(
    env = recordEnv,
    agent = agent(
        id = "testing",
        policy = greedyPolicy(testingQtable)
    ),
    maxStepsPerEpisode = maxStepsPerEpisode,
    callbacks = listOf(
        printEpisodeStart(100)
    )
)
println("Starting testing")
val test = tester.train(testEpisodes)


Comparing the average results between training and testing.

In [ ]:
println("Training Results: ${training.averageReward}")
println("Test Results: ${test.averageReward}")


In [ ]:
val folder = File(recordEnv.folder)
for(file in folder.listFiles()!!.filter { it.isDirectory }) {
    displayVideo(File(folder, file.name))
}

In [ ]:
val format: String = "%6.2f"
val actionSymbols = mapOf(0 to "↑", 1 to "→", 2 to "↓", 3 to "←")
val shape = testingQtable.shape
val stateDf = buildList {
    for (row in 0 until shape[0]) {
        for (col in 0 until shape[1]) {
            val state = intArrayOf(row, col)
            val valueRaw = testingQtable.maxValue(state)
            val valueStr = format.format(valueRaw)
            val action = testingQtable.bestAction(state)
            val arrow = actionSymbols[action] ?: "?"

            // Two rows per cell: one for Value, one for Policy
            add(mapOf("x" to col, "y" to -row, "value" to valueRaw, "label" to valueStr, "type" to "Value"))
            add(mapOf("x" to col, "y" to -row, "value" to valueRaw, "label" to arrow, "type" to "Policy"))
        }
    }
}.flatMap { it.entries }.groupBy({ it.key}, { it.value }).toDataFrame()

plotGrid(stateDf.groupBy("type").map { (typeLabel, group) ->
    group.plot {
        layout {
            title = typeLabel[0]?.toString() ?: ""
            size = 720 to 240
        }


        tiles {
            x("x")
            y("y")
            fillColor("value") {
                scale = continuous(Color.BLUE..Color.WHITE)
            }
            borderLine {
                width = 0.5
                color = Color.BLACK
            }
        }


        text {
            x("x")
            y("y")
            label("label")
            font {
                size = if (typeLabel[0] == "Policy") 8.0 else 3.0
                color = Color.BLACK
            }
        }

        x.axis.name = "x"
        y.axis.name = "y"
    }
})